In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, peak_widths

In [ ]:
data_train = np.loadtxt("ECG5000/ECG5000_TRAIN.txt")
data_test = np.loadtxt("ECG5000/ECG5000_TEST.txt")

df_train = pd.DataFrame(data_train, columns=["class"] + [f"value_{i}" for i in range(1, data_train.shape[1])])
df_test = pd.DataFrame(data_test, columns=["class"] + [f"value_{i}" for i in range(1, data_test.shape[1])])

df_train.head()

In [ ]:
print(f"Total null values: {df_train.isnull().sum().sum()}")
print(f"All values are float64: {df_train.iloc[:, 1:].dtypes.eq('float64').all()}")
y_min = df_train.iloc[:, 1:].min().min()
y_max = df_train.iloc[:, 1:].max().max()
print(f"Min value: {y_min}, Max value: {y_max}")
labels = df_train["class"].unique().astype(int)

In [ ]:
class_counts = df_train["class"].value_counts().sort_index()
class_percentages = df_train["class"].value_counts(normalize=True).sort_index() * 100

class_distribution = pd.DataFrame({
    "Observations": class_counts.tolist() + [class_counts.iloc[1:].sum()],
    "Percentage (%)": class_percentages.tolist() + [class_percentages.iloc[1:].sum()]
}, index=["1/Normal"] + [int(i) for i in class_counts.index[1:]] + ["Abnormal"])

class_distribution.index.name = "Class"
print(class_distribution)

In [ ]:
n_signals = 20

signals_by_class = {
    label: df_train[df_train["class"] == label].iloc[:, 1:].values[:n_signals]
    for label in labels
}

fig, axes = plt.subplots(len(labels), 1, figsize=(12, 3 * len(labels)))

for ax, label in zip(axes, labels):
    signals = signals_by_class[label]
    y = signals.ravel()

    ax.plot(np.arange(y.size), y, linewidth=1)
    ax.set_xlim(0, n_signals * signals.shape[1])
    ax.set_ylim(y_min, y_max)
    ax.set_title(f"Class {label}")
    ax.set_ylabel("Amplitude")
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Sample")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(labels), figsize=(3 * len(labels), 3), sharex=True, sharey=True)

for ax, label in zip(axes, labels):
    signals = signals_by_class[label]

    mean = signals.mean(axis=0)
    std = signals.std(axis=0)
    x = np.arange(len(mean))

    ax.plot(x, mean, label="Mean")
    ax.fill_between(x, mean - std, mean + std, alpha=0.2, label="±1 Std")
    ax.set_xlim(0, len(mean) - 1)
    ax.set_ylim(y_min, y_max)
    ax.set_title(f"Class {label}")
    ax.set_xlabel("Values")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

axes[0].set_ylabel("Amplitude")

plt.tight_layout()
plt.show()

In [ ]:
morphology_stats = []

for label in labels:
    signals = signals_by_class[label]

    morphology_stats.append({
        "Mean Peak Amplitude Max": signals.max(axis=1).mean(),
        "Mean Peak Amplitude Min": signals.min(axis=1).mean(),
    })

morphology_df = pd.DataFrame(
    morphology_stats,
    index=[int(i) for i in labels]
)


abnormal_signals = np.vstack([signals_by_class[label] for label in labels if label != labels[0]])

morphology_df.loc["Abnormal"] = {
    "Mean Peak Amplitude Max": abnormal_signals.max(axis=1).mean(),
    "Mean Peak Amplitude Min": abnormal_signals.min(axis=1).mean(),
}

morphology_df.index = ["1/Normal"] + [int(i) for i in labels[1:]] + ["Abnormal"]
morphology_df.index.name = "Class"
print(morphology_df)

In [ ]:
fig, axes = plt.subplots(1, len(labels), figsize=(3 * len(labels), 3), sharex=True, sharey=True)

for ax, label in zip(axes, labels):
    for signal in signals_by_class[label]:
        ax.plot(signal, alpha=0.25, linewidth=0.8)

    ax.set_title(f"Class {label}")
    ax.set_ylabel("Amplitude")
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Values")
plt.tight_layout()
plt.show()

In [ ]:
peak_stats = []

for label in labels:
    signals = signals_by_class[label]

    peak_counts = []
    peak_width_values = []

    for signal in signals:
        positive_peaks, _ = find_peaks(signal, prominence=1)
        negative_peaks, _ = find_peaks(-signal, prominence=1)

        peaks = np.concatenate([positive_peaks, negative_peaks])
        widths = peak_widths(
            np.concatenate([signal, -signal]), 
            np.concatenate([positive_peaks, negative_peaks]), 
            rel_height=0.5
        )[0] if False else np.concatenate([
            peak_widths(signal, positive_peaks, rel_height=0.5)[0],
            peak_widths(-signal, negative_peaks, rel_height=0.5)[0]
        ])

        peak_counts.append(len(peaks))
        peak_width_values.extend(widths)

    peak_stats.append({
        "Mean Number of Prominent Peaks": np.mean(peak_counts),
        "Mean Peak Width": np.mean(peak_width_values)
    })

peak_df = pd.DataFrame(
    peak_stats,
    index=[int(i) for i in labels]
)

peak_df.index = ["1/Normal"] + [int(i) for i in labels[1:]]
peak_df.index.name = "Class"
print(peak_df)

In [ ]:
fig, axes = plt.subplots(1, len(labels), figsize=(3 * len(labels), 4), sharey=True)

for ax, label in zip(axes, labels):
    signals = signals_by_class[label]

    baselines = signals.mean(axis=1)

    ax.hist(baselines, bins=30, alpha=0.7, orientation="horizontal")
    ax.set_title(f"Class {label}")
    ax.set_xlabel("Observations")
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

axes[0].set_ylabel("Mean amplitude")

plt.tight_layout()
plt.show()